### <p style="text-align: center;"> Análisis exploratorio de datos (EDA) con Spark para información de ventas 2023</p>

#### <p style="text-align: center;"> Tarea 2 – Análisis de Datos, Tendencias Tecnológicas y metodologías.</p>

<p style="text-align: center;"> Nicole Fontecha Lázaro<br>1101760540</p>

<p style="text-align: center;"> Minería para Big Data<br>Grupo 2</p>   

<p style="text-align: center;"> UNIVERSIDAD NACIONAL ABIERTA Y A DISTANCIA - UNAD </p>

<p style="text-align: center;"> Bogotá, March 17th </p>

### Importación de librerías

In [1]:
import findspark 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
import matplotlib.ticker as mticker
from pyspark.sql import SparkSession
from pyspark.sql.functions import max, min, stddev, corr, when, round, to_date
from pyspark.sql.functions import col, avg, sum, count, month, year, dayofweek, quarter

### Conexión a Spark

In [ ]:
findspark.init() #Ubicación automática de Spark

spark = SparkSession.builder \
    .appName("EDA_Ventas2023_UNAD") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate() #Crear conexión con Spark

spark.sparkContext.setLogLevel("ERROR")

print("Conexión a Spark creada exitosamente")


### Data loading

In [ ]:
df = spark.read.csv(                #Carga/Lee el CSV
    "../resources/ventas_2023.csv", #Ruta del archivo
    header=True,                    #Ubica la primera fila como encabezado con el nombre de las columnas
    inferSchema=True                #DEtecta automáticamente los tipos de dato
) 


df.show(3) #Mostrar los primero 3 registros para validar la estructura

In [ ]:
print("Esquema de la data") 
df.printSchema()

print(f"\nDimensions: {df.count()} filas × {len(df.columns)} columnas")
print("Columns:",df.columns)

print("\nEstadísticas descriptivas básicas:") 
df.describe().show()

### Análisis exploratorio de datos (EDA – Exploratory Data Analysis)

##### Identificación de ruido

In [ ]:
print("Valores nulos por columna:")
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

total      = df.count()
sin_duplic = df.dropDuplicates().count()    #Conteo de registros únicos
duplicados = total - sin_duplic             #Cálculo de registros duplicados
print(f"\nRegistros duplicados: {duplicados} ({duplicados/total*100:.2f}%)")

invalidos = df.filter(col("monto") < 0).count() #Cálculo de valores inválidos (monto negativo)
print(f"\nRegistros con monto < 0: {invalidos}")

print("\nRango de tiempo analizado:")
df.select(
    min("fecha").alias("fecha_inicio"),
    max("fecha").alias("fecha_fin")
).show()

print("Valores únicos por variable categórica")
for c in ["producto", "region", "tipo_pago", "dia_semana"]:
    vals = [r[c] for r in df.select(c).distinct().collect()]
    print(f"{c}: {vals}")


In [ ]:
# Método IQR (Rango Intercuartílico) sobre Spark para la variable 'monto'
Q1, Q3 = df.approxQuantile("monto", [0.25, 0.75], 0.01)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers = df.filter(
    (col("monto") < lim_inf) | (col("monto") > lim_sup))
n_out = outliers.count()

print(f"Análisis de Outliers (Método IQR) para la variable 'monto':")
print(f"   Q1          : ${Q1:,.2f}")
print(f"   Q3          : ${Q3:,.2f}")
print(f"   IQR         : ${IQR:,.2f}")
print(f"   Límite inf. : ${lim_inf:,.2f}")
print(f"   Límite sup. : ${lim_sup:,.2f}")
print(f"   Outliers    : {n_out} ({n_out/df.count()*100:.2f}%)")

if n_out > 0:
    print("\nOutliers detectados:")
    outliers.select("fecha", "monto", "producto",
                    "region", "tipo_pago").show()

##### Identificación de patrones

In [ ]:
df.createOrReplaceTempView("ventas")

print("=" * 60)
print("  RESUMEN DE PATRONES IDENTIFICADOS – 2023")
print("=" * 60)

#Mes con mayor volumen
print("\nTop 3 meses por ventas totales:")
spark.sql("""
    SELECT mes,
           ROUND(SUM(monto), 2)  AS ventas_totales,
           ROUND(AVG(monto), 2)  AS venta_promedio
    FROM ventas
    GROUP BY mes
    ORDER BY ventas_totales DESC
    LIMIT 3
""").show()

#Producto más vendido
print("Ventas por producto:")
spark.sql("""
    SELECT producto,
           COUNT(*)              AS dias,
           ROUND(SUM(monto), 2)  AS total
    FROM ventas
    GROUP BY producto
    ORDER BY total DESC
""").show()

#Región más activa
print("Ventas por región:")
spark.sql("""
    SELECT region,
           ROUND(SUM(monto), 2)  AS total,
           ROUND(AVG(monto), 2)  AS promedio
    FROM ventas
    GROUP BY region
    ORDER BY total DESC
""").show()

#Método de pago preferido
print("Ventas por tipo de pago:")
spark.sql("""
    SELECT tipo_pago,
           COUNT(*)              AS registros,
           ROUND(SUM(monto), 2)  AS total
    FROM ventas
    GROUP BY tipo_pago
    ORDER BY total DESC
""").show()

##### Análisis visual

In [ ]:
#Agregar nuevas columnas temporales
df = df.withColumn("trimestre", quarter("fecha")) \
       .withColumn("num_dia_semana", dayofweek("fecha")) \
       .withColumn("nombre_mes",
           when(col("mes") == 1,  "Enero")
          .when(col("mes") == 2,  "Febrero")
          .when(col("mes") == 3,  "Marzo")
          .when(col("mes") == 4,  "Abril")
          .when(col("mes") == 5,  "Mayo")
          .when(col("mes") == 6,  "Junio")
          .when(col("mes") == 7,  "Julio")
          .when(col("mes") == 8,  "Agosto")
          .when(col("mes") == 9,  "Septiembre")
          .when(col("mes") == 10, "Octubre")
          .when(col("mes") == 11, "Noviembre")
          .otherwise("Diciembre")) \
       .withColumn("ingreso_total",
           round(col("monto") * col("cantidad"), 2))

print("Columnas agregadas: trimestre, nombre_mes, ingreso_total")
df.show(3)

In [ ]:
#TENDENCIA MENSUAL

#Generar nuevo df con data agrupada por mes
ventas_mes = df.groupBy("mes", "nombre_mes") \
    .agg(
        round(sum("monto"),         2).alias("ventas_totales"),
        round(avg("monto"),         2).alias("venta_promedio"),
        round(sum("ingreso_total"), 2).alias("ingreso_total"),
        round(count("*"),           0).alias("dias_con_venta"),
        round(stddev("monto"),      2).alias("desv_estandar")
    ) \
    .orderBy("mes") \
    .toPandas()

#Gráfica de tendencia mensual doble eje
fig, ax1 = plt.subplots(figsize=(13, 5))

ax1.fill_between(ventas_mes["nombre_mes"],      #Graficar Ventas Totales en el primer eje Y
                 ventas_mes["ventas_totales"],  #fill_between crea el área sombreada bajo la curva
                 alpha=0.2, color="#041427")
ax1.plot(ventas_mes["nombre_mes"], ventas_mes["ventas_totales"],
         marker="o", color="#1565c0", linewidth=2.5, markersize=6,
         label="Ventas totales (monto)")

ax2 = ax1.twinx()  #Creación del segundo eje Y que comparte el mismo eje X
ax2.plot(ventas_mes["nombre_mes"], ventas_mes["ingreso_total"],
         marker="s", color="#e65100", linewidth=2,
         linestyle="--", markersize=5, label="Ingreso total (monto × cantidad)")

ax1.set_title("Tendencia Mensual de Ventas e Ingresos – 2023", #Ajuste de título
              fontsize=13, fontweight="bold")
ax1.set_xlabel("Mes")                                          #Ajuste de etiquetas
ax1.set_ylabel("Ventas totales ($)", color="#1565c0")
ax2.set_ylabel("Ingreso total ($)", color="#e65100")

ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}")) #Agregar formatos a valores
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}")) #de los eje Y

plt.xticks(rotation=35, ha="right") #Rotación de las etiquetas del eje X para mejorar la lectura

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left") #Combinación de leyendas en una sola

plt.tight_layout() #Ajuste automático de los elementos
plt.show()

In [ ]:
#DISTRIBUCIÓN POR TRIMESTRE

#Generar nuevo df con data agrupada por trimestre
ventas_trim = df.groupBy("trimestre") \
    .agg(
        round(sum("monto"),  2).alias("ventas_totales"),
        round(avg("monto"),  2).alias("venta_promedio"),
        count("*").alias("dias")
    ) \
    .orderBy("trimestre") \
    .toPandas()

ventas_trim["etiqueta"] = "Q" + ventas_trim["trimestre"].astype(str)

#Gráfica de barras por trimestre
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(ventas_trim["etiqueta"], ventas_trim["ventas_totales"],   #Creación del gráfico de barras
              color=sns.color_palette("Blues_d", 4),                    #Paleta de colores degradada Blues_d
              edgecolor="white", linewidth=0.8, width=0.5)


for bar in bars:                            #Agregar etoquetas sobre cada barra
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + ventas_trim["ventas_totales"].max() * 0.01,
            f"${bar.get_height():,.0f}",    #Formato
            ha="center", va="bottom",       #Alineación centrada y sobre la barra
            fontsize=9, fontweight="bold")

ax.set_title("Ventas Totales por Trimestre – 2023",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Trimestre")
ax.set_ylabel("Ventas ($)")

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}")) #Agregar formato a valores del eje Y

plt.tight_layout()
plt.show()

In [ ]:
#VENTAS POR DÍA DE LA SEMANA

#Generar nuevo df con data agrupada por día de la semana
ventas_dia = df.groupBy("dia_semana") \
    .agg(
        round(avg("monto"),  2).alias("venta_promedio"),
        round(sum("monto"),  2).alias("ventas_totales"),
        count("*").alias("num_dias")
    ) \
    .toPandas()

# Ordenar el dataset
orden_dias = ["Lunes","Martes","Miércoles","Jueves", "Viernes","Sábado","Domingo"]
ventas_dia["dia_semana"] = pd.Categorical(ventas_dia["dia_semana"], categories=orden_dias, ordered=True)
ventas_dia = ventas_dia.sort_values("dia_semana")

#Gráfica de barras por ciclo semanal
fig, axes = plt.subplots(1, 2, figsize=(14, 5)) #Plot con 2 subgráficos

#Gráfico de la izquierda - Venta Promedio
axes[0].bar(ventas_dia["dia_semana"], ventas_dia["venta_promedio"],
            color=sns.color_palette("coolwarm", 7),
            edgecolor="white")
axes[0].set_title("Venta Promedio por Día de la Semana",fontweight="bold") #Título para el primer eje
axes[0].set_ylabel("Monto promedio ($)")        #Etiqueta del eje vertical
axes[0].tick_params(axis="x", rotation=35)      #Inclina los nombres de los días

#Gráfico de la derecha - Frecuencia)
axes[1].bar(ventas_dia["dia_semana"], ventas_dia["num_dias"],
            color=sns.color_palette("YlOrRd", 7),        
            edgecolor="white")                           
axes[1].set_title("Frecuencia de Registros por Día",fontweight="bold")
axes[1].set_ylabel("N° de días registrados")
axes[1].tick_params(axis="x", rotation=35)

plt.suptitle("Comportamiento de Ventas por Día de la Semana", fontsize=13, fontweight="bold") #Título general
plt.tight_layout()
plt.show() 

In [ ]:
#GRÁFICAS DE DISTRIBUCIÓN

montos = df.select("monto", "cantidad", "ingreso_total").toPandas()

#Gráfico de histograma, boxplot y barras de distribución
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

#Histograma KDE
axes[0].hist(montos["monto"], bins=35, color="#1565c0",
             edgecolor="white", alpha=0.8, density=True)
montos["monto"].plot.kde(ax=axes[0], color="red",
                          linewidth=2, label="KDE")
axes[0].axvline(montos["monto"].mean(), color="orange",
                linestyle="--", label=f'Media: ${montos["monto"].mean():,.0f}') #Agrega línea de promedio
axes[0].set_title("Distribución de Monto Diario", fontweight="bold")
axes[0].set_xlabel("Monto ($)")
axes[0].legend(fontsize=8)

# Boxplot de monto
axes[1].boxplot(montos["monto"].dropna(), patch_artist=True,
                boxprops=dict(facecolor="#bbdefb", color="#1565c0"),
                medianprops=dict(color="#e65100", linewidth=2.5),
                flierprops=dict(marker="o", markersize=4,
                                markerfacecolor="#e65100", alpha=0.5))
axes[1].set_title("Boxplot – Monto Diario", fontweight="bold")
axes[1].set_ylabel("Monto ($)")

# Distribución de cantidad
axes[2].hist(montos["cantidad"], bins=10,
             color="#2e7d4f", edgecolor="white", alpha=0.85)
axes[2].set_title("Distribución de Cantidad Vendida", fontweight="bold")
axes[2].set_xlabel("Unidades")
axes[2].set_ylabel("Frecuencia")

plt.suptitle("Análisis de Distribución de Variables Numéricas",fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#DISTRIBUCIÓN POR PRODUCTO

#Generar nuevo df con data agrupada por producto
ventas_prod = df.groupBy("producto") \
    .agg(
        round(sum("monto"),         2).alias("ventas_totales"),
        round(avg("monto"),         2).alias("venta_promedio"),
        round(sum("ingreso_total"), 2).alias("ingreso_total"),
        count("*").alias("dias_registrados")
    ) \
    .orderBy(col("ventas_totales").desc()) \
    .toPandas()

#Gráfico de barras horizontales y pie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#Barras horizontales
colors = sns.color_palette("tab10", len(ventas_prod))
axes[0].barh(ventas_prod["producto"], ventas_prod["ventas_totales"],
             color=colors)
axes[0].set_title("Ventas Totales por Producto", fontweight="bold")
axes[0].set_xlabel("Ventas ($)")
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))

#Pie
axes[1].pie(ventas_prod["ventas_totales"],
            labels=ventas_prod["producto"],
            autopct="%1.1f%%", colors=colors,
            startangle=140, pctdistance=0.82)
axes[1].set_title("Participación por Producto", fontweight="bold")

plt.suptitle("Análisis de Ventas por Producto",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#DISTRIBUCIÓN POR REGIOS Y TIPO DE PAGO

#Generar nuevo df con data agrupada por region
ventas_reg = df.groupBy("region") \
    .agg(
        round(sum("monto"),  2).alias("ventas_totales"),
        round(avg("monto"),  2).alias("venta_promedio"),
        count("*").alias("registros")
    ) \
    .orderBy(col("ventas_totales").desc()) \
    .toPandas()

#Generar nuevo df con data agrupada por tipo de pago
ventas_pago = df.groupBy("tipo_pago") \
    .agg(
        round(sum("monto"),  2).alias("ventas_totales"),
        count("*").alias("registros")
    ) \
    .orderBy(col("ventas_totales").desc()) \
    .toPandas()

#Gráficos de barras verticales
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#Gráfico por region
sns.barplot(data=ventas_reg, x="region", y="ventas_totales", hue="region",
            palette="Blues_d", ax=axes[0])
axes[0].set_title("Ventas Totales por Región", fontweight="bold")
axes[0].set_xlabel("Región")
axes[0].set_ylabel("Ventas ($)")
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))

#Gráfico por tipo de pago
sns.barplot(data=ventas_pago, x="tipo_pago", y="ventas_totales", hue="tipo_pago",
            palette="Greens_d", ax=axes[1])
axes[1].set_title("Ventas Totales por Tipo de Pago", fontweight="bold")
axes[1].set_xlabel("Tipo de pago")
axes[1].set_ylabel("Ventas ($)")
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))

plt.suptitle("Análisis por Variables Categóricas",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#Correlación Spark entre monto y cantidad
r_spark = df.stat.corr("monto", "cantidad")
print(f"Índice de correlación entre monto y cantidad: {r_spark:.4f}")

#Relación entre variables
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

#Mapa de calor de relación entre variables numéricas
df_num = df.select("monto", "cantidad", "mes", "trimestre", "ingreso_total").toPandas()
corr_matrix = df_num.corr()
sns.heatmap(corr_matrix, annot=True, fmt=".3f",
            cmap="coolwarm", center=0, square=True,
            linewidths=0.8, ax=axes[0],
            cbar_kws={"shrink": 0.75})
axes[0].set_title("Mapa de Correlaciones", fontweight="bold")

#Gráfico scatter de monto vs cantidad
axes[1].scatter(df_num["cantidad"], df_num["monto"],
                c=df_num["mes"], cmap="plasma",
                alpha=0.6, s=30, edgecolors="none")
sc = axes[1].scatter(df_num["cantidad"], df_num["monto"],
                     c=df_num["mes"], cmap="plasma",
                     alpha=0.6, s=30)
plt.colorbar(sc, ax=axes[1], label="Mes")
axes[1].set_title("Monto vs. Cantidad\n(color = Mes)",
                  fontweight="bold")
axes[1].set_xlabel("Cantidad de unidades")
axes[1].set_ylabel("Monto ($)")

plt.suptitle("Análisis de Correlaciones entre Variables",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#Generar nuevo df con data agrupada por mes y día de la semana
heat_data = df.groupBy("mes", "dia_semana") \
    .agg(round(avg("monto"), 2).alias("monto_prom")) \
    .toPandas()

orden = ["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"]
pivot_heat = heat_data.pivot(index="dia_semana", columns="mes", values="monto_prom")
pivot_heat = pivot_heat.reindex([d for d in orden if d in pivot_heat.index])
pivot_heat.columns = ["Ene","Feb","Mar","Abr","May","Jun", "Jul","Ago","Sep","Oct","Nov","Dic"]

#Mapa de calor de promedio de ventas por día de la semana y mes
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_heat, annot=True, fmt=".0f",
            cmap="YlOrRd", linewidths=0.5, ax=ax,
            cbar_kws={"label": "Monto promedio ($)"})
ax.set_title("Mapa de Calor: Monto Promedio por Día de la Semana y Mes",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Mes")
ax.set_ylabel("Día de la semana")
plt.tight_layout()
plt.show()

##### Cierre de sesión de Spark

In [ ]:
spark.stop()
print("Sesión de Spark cerrada correctamente")
